In [35]:
import pandas as pd 
import numpy as np
import tensorflow as tf 
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import DenseNet121

In [12]:
df_train = pd.read_csv(r'../data/selected_col/model_train.csv')
df_test = pd.read_csv(r'../data/selected_col/model_test.csv')
df_val = pd.read_csv(r'../data/selected_col/model_val.csv')

In [8]:
df_train

,Unnamed: 0,lesion_id,image_id,dx,dx_type,age,sex,localization,path,dx_encode
0,8561,HAM_0007001,ISIC_0026573,nv,histo,60.0,female,lower extremity,../data/all_image/ISIC_0026573.jpg,5
1,6583,HAM_0006055,ISIC_0031456,nv,follow_up,75.0,female,lower extremity,../data/all_image/ISIC_0031456.jpg,5
2,5171,HAM_0006068,ISIC_0024591,nv,follow_up,35.0,male,back,../data/all_image/ISIC_0024591.jpg,5
3,3248,HAM_0001590,ISIC_0025722,nv,follow_up,50.0,male,lower extremity,../data/all_image/ISIC_0025722.jpg,5
4,7903,HAM_0002015,ISIC_0034035,nv,histo,30.0,female,lower extremity,../data/all_image/ISIC_0034035.jpg,5
...,...,...,...,...,...,...,...,...,...,...
7004,7810,HAM_0000639,ISIC_0025257,nv,histo,70.0,female,upper extremity,../data/all_image/ISIC_0025257.jpg,5
7005,853,HAM_0007043,ISIC_0029947,bkl,consensus,75.0,male,face,../data/all_image/ISIC_0029947.jpg,2
7006,4257,HAM_0001568,ISIC_0030614,nv,follow_up,50.0,male,trunk,../data/all_image/ISIC_0030614.jpg,5
7007,3382,HAM_0005800,ISIC_0028864,nv,follow_up,40.0,female,upper extremity,../data/all_image/ISIC_0028864.jpg,5


In [9]:
df_test

,Unnamed: 0,lesion_id,image_id,dx,dx_type,age,sex,localization,path,dx_encode
0,9606,HAM_0000627,ISIC_0034019,nv,consensus,50.0,unknown,unknown,../data/all_image/ISIC_0034019.jpg,5
1,5661,HAM_0001761,ISIC_0026732,nv,follow_up,30.0,male,abdomen,../data/all_image/ISIC_0026732.jpg,5
2,9355,HAM_0002726,ISIC_0025473,nv,consensus,5.0,male,foot,../data/all_image/ISIC_0025473.jpg,5
3,7757,HAM_0003424,ISIC_0033552,nv,histo,45.0,male,back,../data/all_image/ISIC_0033552.jpg,5
4,2284,HAM_0004081,ISIC_0031957,mel,histo,70.0,female,lower extremity,../data/all_image/ISIC_0031957.jpg,4
...,...,...,...,...,...,...,...,...,...,...
1498,4063,HAM_0006808,ISIC_0027438,nv,follow_up,65.0,male,lower extremity,../data/all_image/ISIC_0027438.jpg,5
1499,9437,HAM_0002425,ISIC_0032682,nv,consensus,20.0,male,back,../data/all_image/ISIC_0032682.jpg,5
1500,6604,HAM_0003747,ISIC_0026582,nv,follow_up,50.0,male,trunk,../data/all_image/ISIC_0026582.jpg,5
1501,5841,HAM_0002807,ISIC_0025050,nv,follow_up,45.0,male,trunk,../data/all_image/ISIC_0025050.jpg,5


In [10]:
df_val

,Unnamed: 0,lesion_id,image_id,dx,dx_type,age,sex,localization,path,dx_encode
0,8083,HAM_0005421,ISIC_0028060,nv,histo,75.0,male,trunk,../data/all_image/ISIC_0028060.jpg,5
1,4277,HAM_0007328,ISIC_0030684,nv,follow_up,50.0,male,lower extremity,../data/all_image/ISIC_0030684.jpg,5
2,4773,HAM_0001676,ISIC_0031587,nv,follow_up,60.0,male,upper extremity,../data/all_image/ISIC_0031587.jpg,5
3,7783,HAM_0007494,ISIC_0033215,nv,histo,25.0,female,face,../data/all_image/ISIC_0033215.jpg,5
4,3011,HAM_0001180,ISIC_0029303,nv,follow_up,50.0,male,trunk,../data/all_image/ISIC_0029303.jpg,5
...,...,...,...,...,...,...,...,...,...,...
1498,7310,HAM_0003507,ISIC_0033459,nv,histo,30.0,male,lower extremity,../data/all_image/ISIC_0033459.jpg,5
1499,413,HAM_0005514,ISIC_0024453,bkl,histo,65.0,female,face,../data/all_image/ISIC_0024453.jpg,2
1500,1962,HAM_0001986,ISIC_0024999,mel,histo,50.0,male,upper extremity,../data/all_image/ISIC_0024999.jpg,4
1501,1053,HAM_0003982,ISIC_0030056,bkl,consensus,50.0,male,trunk,../data/all_image/ISIC_0030056.jpg,2


In [3]:
img_size = (224,224)
batch_size = 32

In [14]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(factor=0.15, value_range=(0.0, 1.0)),
])

In [5]:
def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [6]:
def preprocess_train(image_path, label):
    image, label = preprocess_image(image_path, label)
    image = data_augmentation(image)
    return image, label

In [7]:
def preprocess_test(image_path, label):
    image, label = preprocess_image(image_path, label)
    return image, label

In [8]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset = (
    train_dataset
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [9]:
val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset = (
    val_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [10]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset = (
    test_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [11]:
y_train = df_train["dx_encode"].values


class_weights = compute_class_weight(class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train)

# Convert to dictionary
class_weight_dict = dict(enumerate(class_weights))

class_weight_dict

{0: np.float64(4.372426699937617),
 1: np.float64(2.7813492063492062),
 2: np.float64(1.3020620471855842),
 3: np.float64(12.51607142857143),
 4: np.float64(1.2853475151292866),
 5: np.float64(0.2133572798392743),
 6: np.float64(10.113997113997113)}

In [13]:
model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")

])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(train_dataset, epochs=10, validation_data=val_dataset,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/backend/tensorflow/nn.py:1402: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


220/220 ━━━━━━━━━━━━━━━━━━━━ 194s 850ms/step - accuracy: 0.3789 - loss: 1.9500 - val_accuracy: 0.2202 - val_loss: 1.7286
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 196s 852ms/step - accuracy: 0.2617 - loss: 1.7744 - val_accuracy: 0.4731 - val_loss: 1.3861
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 862ms/step - accuracy: 0.3628 - loss: 1.6875 - val_accuracy: 0.4385 - val_loss: 1.6049
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 792ms/step - accuracy: 0.4170 - loss: 1.6277 - val_accuracy: 0.3566 - val_loss: 1.5999
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 855ms/step - accuracy: 0.3493 - loss: 1.5634 - val_accuracy: 0.3560 - val_loss: 1.6841
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 237s 1s/step - accuracy: 0.4704 - loss: 1.4785 - val_accuracy: 0.2761 - val_loss: 1.9040
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 222s 982ms/step - accuracy: 0.3855 - loss: 1.5499 - val_accuracy: 0.5223 - val_loss: 1.2231
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 190s 840ms/step - accuracy: 0.4500 - loss: 1.5040 

In [25]:
test_loss,test_accuracy = model.evaluate(test_dataset)
print(f'test accuracy : {test_accuracy}')
print(f'test loss : {test_loss}')

47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 187ms/step - accuracy: 0.4864 - loss: 1.3443
test accuracy : 0.4863606095314026
test loss : 1.3443496227264404


In [ ]:
#deep cnn

In [25]:
deep_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(256,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(512,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])
deep_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_18 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 10, 10, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 5, 5, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │     3,277,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,847,431 (18.49 MB)

 Trainable params: 4,847,431 (18.49 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
deep_model.compile(optimizer='adam',
              loss='SparseCategoricalCrossentropy',
              metrics=['accuracy'])

In [27]:
history = deep_model.fit(train_dataset, epochs=10, validation_data=val_dataset,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 247s 1s/step - accuracy: 0.1605 - loss: 1.9425 - val_accuracy: 0.1098 - val_loss: 1.9630
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 245s 1s/step - accuracy: 0.0468 - loss: 1.9465 - val_accuracy: 0.1098 - val_loss: 1.9530
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 268s 1s/step - accuracy: 0.1097 - loss: 1.9464 - val_accuracy: 0.1098 - val_loss: 1.9507
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 253s 1s/step - accuracy: 0.0832 - loss: 1.9464 - val_accuracy: 0.0120 - val_loss: 1.9466
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 247s 1s/step - accuracy: 0.0609 - loss: 1.9464 - val_accuracy: 0.1098 - val_loss: 1.9452
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 248s 1s/step - accuracy: 0.1900 - loss: 1.9463 - val_accuracy: 0.0120 - val_loss: 1.9428
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 239s 1s/step - accuracy: 0.1024 - loss: 1.9464 - val_accuracy: 0.1098 - val_loss: 1.9424
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 243s 1s/step - accuracy: 0.5346 - loss: 1.9462 - val_accuracy: 0.669

In [15]:
deep_model1 = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
        layers.MaxPooling2D(2,2),

    layers.Conv2D(256,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])
deep_model1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
deep_model1.compile(optimizer='adam',
              loss='SparseCategoricalCrossentropy',
              metrics=['accuracy'])

In [30]:
history = deep_model1.fit(train_dataset, epochs=10, validation_data=val_dataset,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 214s 956ms/step - accuracy: 0.2926 - loss: 1.9307 - val_accuracy: 0.2562 - val_loss: 2.1183
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 211s 935ms/step - accuracy: 0.3744 - loss: 1.8573 - val_accuracy: 0.2994 - val_loss: 1.7553
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 203s 901ms/step - accuracy: 0.3604 - loss: 1.7751 - val_accuracy: 0.2302 - val_loss: 1.9924
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 217s 961ms/step - accuracy: 0.3768 - loss: 1.7251 - val_accuracy: 0.5635 - val_loss: 1.2884
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 226s 1s/step - accuracy: 0.4459 - loss: 1.5843 - val_accuracy: 0.4172 - val_loss: 1.6291
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 207s 919ms/step - accuracy: 0.4514 - loss: 1.5537 - val_accuracy: 0.5077 - val_loss: 1.4347
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 207s 918ms/step - accuracy: 0.4867 - loss: 1.4851 - val_accuracy: 0.4578 - val_loss: 1.4160
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 210s 930ms/step - accuracy: 0.4861 - loss: 1.4563 

In [17]:
test_loss,test_accuracy = deep_model1.evaluate(test_dataset)
print(f'test accuracy : {test_accuracy}')
print(f'test loss : {test_loss}')

47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 190ms/step - accuracy: 0.6560 - loss: 1.9103
test accuracy : 0.6560212969779968
test loss : 1.9103336334228516


In [24]:
test_loss, test_acc = deep_model1.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 252ms/step - accuracy: 0.6560 - loss: 1.9103


In [25]:
test_loss,test_acc

(1.9103336334228516, 0.6560212969779968)

In [ ]:
'''batch normalization cnn'''

In [19]:
bn_cnn_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7,activation='softmax')

])

bn_cnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 52, 52, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,247,143 (84.87 MB)

 Trainable params: 22,246,695 (84.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [21]:
bn_cnn_model.compile(optimizer='adam',
              loss='SparseCategoricalCrossentropy',
              metrics=['accuracy'])

In [22]:
history = bn_cnn_model.fit(train_dataset, epochs=10, validation_data=val_dataset,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.1465 - loss: 6.5308 - val_accuracy: 0.0120 - val_loss: 3.3440
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.4785 - loss: 1.8590 - val_accuracy: 0.2502 - val_loss: 2.9600
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 2917s 13s/step - accuracy: 0.5888 - loss: 1.9264 - val_accuracy: 0.5802 - val_loss: 1.9203
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 302s 1s/step - accuracy: 0.6136 - loss: 1.8688 - val_accuracy: 0.6414 - val_loss: 1.8566
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.6099 - loss: 1.8529 - val_accuracy: 0.6168 - val_loss: 1.8767
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 306s 1s/step - accuracy: 0.6021 - loss: 1.7739 - val_accuracy: 0.6208 - val_loss: 1.8964
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 310s 1s/step - accuracy: 0.6265 - loss: 1.7483 - val_accuracy: 0.6108 - val_loss: 1.8373
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.6212 - loss: 1.7352 - val_accuracy: 0.6

In [23]:
test_loss,test_accuracy = bn_cnn_model.evaluate(test_dataset)
print(f'test accuracy : {test_accuracy}')
print(f'test loss : {test_loss}')

47/47 ━━━━━━━━━━━━━━━━━━━━ 13s 273ms/step - accuracy: 0.5057 - loss: 1.7003
test accuracy : 0.5056553483009338
test loss : 1.700343132019043


In [ ]:
'''cnn dropout'''

In [27]:
dropout_cnn_model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation='softmax')

])

dropout_cnn_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,863 (42.61 MB)

 Trainable params: 11,169,863 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
dropout_cnn_model.compile(optimizer='adam',
              loss='SparseCategoricalCrossentropy',
              metrics=['accuracy'])

In [29]:
history = dropout_cnn_model.fit(train_dataset, epochs=10, validation_data=val_dataset,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 194s 854ms/step - accuracy: 0.3067 - loss: 1.9599 - val_accuracy: 0.1550 - val_loss: 1.9518
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 192s 848ms/step - accuracy: 0.1535 - loss: 1.9099 - val_accuracy: 0.2202 - val_loss: 1.8063
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 180s 796ms/step - accuracy: 0.2607 - loss: 1.8812 - val_accuracy: 0.3460 - val_loss: 1.6932
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 165s 728ms/step - accuracy: 0.1151 - loss: 1.9780 - val_accuracy: 0.0512 - val_loss: 2.0527
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 165s 727ms/step - accuracy: 0.0380 - loss: 1.9483 - val_accuracy: 0.0126 - val_loss: 1.9722
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 165s 730ms/step - accuracy: 0.0742 - loss: 1.9506 - val_accuracy: 0.0326 - val_loss: 1.9554
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 161s 712ms/step - accuracy: 0.1096 - loss: 1.9477 - val_accuracy: 0.0120 - val_loss: 1.9627
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 161s 712ms/step - accuracy: 0.1204 - loss: 1.94

In [ ]:
'''according to the test accuray and report this underfit'''

In [34]:
dropout_cnn_model1 = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation='softmax')

])

dropout_cnn_model1.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_16 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
dropout_cnn_model1.compile(optimizer='adam',
              loss='SparseCategoricalCrossentropy',
              metrics=['accuracy'])

In [36]:
history = dropout_cnn_model1.fit(train_dataset, epochs=10, validation_data=val_dataset,class_weight=class_weight_dict)

Epoch 1/10


220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 852ms/step - accuracy: 0.3043 - loss: 1.9593 - val_accuracy: 0.3513 - val_loss: 1.8882
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 189s 833ms/step - accuracy: 0.3767 - loss: 1.8370 - val_accuracy: 0.1331 - val_loss: 2.0712
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 810ms/step - accuracy: 0.2368 - loss: 1.9166 - val_accuracy: 0.1464 - val_loss: 1.8106
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 818ms/step - accuracy: 0.2408 - loss: 1.8115 - val_accuracy: 0.2289 - val_loss: 1.7554
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 831ms/step - accuracy: 0.2898 - loss: 1.7398 - val_accuracy: 0.3087 - val_loss: 1.8003
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 831ms/step - accuracy: 0.3230 - loss: 1.6829 - val_accuracy: 0.4305 - val_loss: 1.5592
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 856ms/step - accuracy: 0.3905 - loss: 1.6056 - val_accuracy: 0.1843 - val_loss: 1.7808
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 788ms/step - accuracy: 0.3848 - loss: 1.59

In [37]:
test_loss,test_accuracy = dropout_cnn_model1.evaluate(test_dataset)
print(f'test accuracy : {test_accuracy}')
print(f'test loss : {test_loss}')

47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 173ms/step - accuracy: 0.4697 - loss: 1.3509
test accuracy : 0.46972721815109253
test loss : 1.3509488105773926


In [ ]:
'''mobile net v2'''

In [39]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

mobilenet_model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

mobilenet_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [40]:
mobilenet_model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [41]:
history = mobilenet_model.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 142s 611ms/step - accuracy: 0.3701 - loss: 1.7676 - val_accuracy: 0.4857 - val_loss: 1.4373
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 141s 617ms/step - accuracy: 0.4670 - loss: 1.4904 - val_accuracy: 0.4331 - val_loss: 1.3176
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 145s 635ms/step - accuracy: 0.4219 - loss: 1.4523 - val_accuracy: 0.5589 - val_loss: 1.2276
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 145s 631ms/step - accuracy: 0.4411 - loss: 1.3488 - val_accuracy: 0.6181 - val_loss: 1.0373
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 142s 620ms/step - accuracy: 0.4406 - loss: 1.3256 - val_accuracy: 0.5183 - val_loss: 1.2084
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 142s 622ms/step - accuracy: 0.4875 - loss: 1.2711 - val_accuracy: 0.5329 - val_loss: 1.2024
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 142s 622ms/step - accuracy: 0.4745 - loss: 1.2719 - val_accuracy: 0.5702 - val_loss: 1.0843
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 148s 646ms/step - accuracy: 0.4784 - loss: 1.24

In [42]:
test_loss, test_accuracy = mobilenet_model.evaluate(test_dataset)

print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 21s 445ms/step - accuracy: 0.6134 - loss: 0.9681
Test Accuracy : 0.6134397983551025
Test Loss : 0.9681494235992432


In [ ]:
'''ecifeintnetB0'''

In [45]:
ef_base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model.trainable = False

efficientnet_model = models.Sequential([

    ef_base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

efficientnet_model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 10s 1us/step


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [46]:
efficientnet_model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [47]:
history = efficientnet_model.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 198s 841ms/step - accuracy: 0.0946 - loss: 1.9665 - val_accuracy: 0.0140 - val_loss: 1.9521
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 190s 835ms/step - accuracy: 0.0334 - loss: 1.9463 - val_accuracy: 0.0512 - val_loss: 1.9500
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 2292s 10s/step - accuracy: 0.0156 - loss: 1.9478 - val_accuracy: 0.0140 - val_loss: 1.9479
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 180s 792ms/step - accuracy: 0.0278 - loss: 1.9462 - val_accuracy: 0.1098 - val_loss: 1.9484
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 796ms/step - accuracy: 0.0656 - loss: 1.9464 - val_accuracy: 0.1098 - val_loss: 1.9455
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 177s 780ms/step - accuracy: 0.0354 - loss: 1.9463 - val_accuracy: 0.0120 - val_loss: 1.9458
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 815ms/step - accuracy: 0.1043 - loss: 1.9463 - val_accuracy: 0.1098 - val_loss: 1.9466
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 177s 782ms/step - accuracy: 0.0330 - loss: 1.946

In [48]:
ef_base_model1 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model1.trainable = False

efficientnet_model1 = models.Sequential([

    ef_base_model1,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model1.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [49]:
efficientnet_model1.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [50]:
history = efficientnet_model1.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 320s 1s/step - accuracy: 0.1803 - loss: 1.9683 - val_accuracy: 0.0120 - val_loss: 1.9476
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 815ms/step - accuracy: 0.0582 - loss: 1.9470 - val_accuracy: 0.0120 - val_loss: 1.9449
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 198s 879ms/step - accuracy: 0.0983 - loss: 1.9463 - val_accuracy: 0.0120 - val_loss: 1.9450
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 196s 869ms/step - accuracy: 0.0869 - loss: 1.9463 - val_accuracy: 0.0120 - val_loss: 1.9459
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 829ms/step - accuracy: 0.1557 - loss: 1.9463 - val_accuracy: 0.6693 - val_loss: 1.9429
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 183s 809ms/step - accuracy: 0.0892 - loss: 1.9464 - val_accuracy: 0.6693 - val_loss: 1.9431
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 863ms/step - accuracy: 0.2441 - loss: 1.9464 - val_accuracy: 0.0120 - val_loss: 1.9423
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 189s 831ms/step - accuracy: 0.4610 - loss: 1.9462 

In [ ]:
test_loss,test_accuracy = efficientnet_model1.evaluate(test_dataset)
print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 29s 608ms/step - accuracy: 0.0512 - loss: 1.9480
Test Accuracy : 1.9479985237121582
Test Loss : 0.05123087018728256


In [52]:
ef_base_model2 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model2.trainable = False

efficientnet_model2 = models.Sequential([

    ef_base_model2,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model2.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [53]:
efficientnet_model2.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [54]:
history = efficientnet_model2.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 202s 869ms/step - accuracy: 0.0839 - loss: 1.9773 - val_accuracy: 0.0120 - val_loss: 1.9877
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 819ms/step - accuracy: 0.0877 - loss: 1.9466 - val_accuracy: 0.1111 - val_loss: 1.9438
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 194s 854ms/step - accuracy: 0.2157 - loss: 1.9464 - val_accuracy: 0.1111 - val_loss: 1.9429
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 831ms/step - accuracy: 0.0312 - loss: 1.9463 - val_accuracy: 0.0326 - val_loss: 1.9486
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 205s 905ms/step - accuracy: 0.0983 - loss: 1.9471 - val_accuracy: 0.0326 - val_loss: 1.9479
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 191s 847ms/step - accuracy: 0.0384 - loss: 1.9463 - val_accuracy: 0.1098 - val_loss: 1.9448
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 211s 937ms/step - accuracy: 0.0836 - loss: 1.9462 - val_accuracy: 0.1098 - val_loss: 1.9461
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 199s 876ms/step - accuracy: 0.0658 - loss: 1.94

In [55]:
test_loss,test_accuracy = efficientnet_model2.evaluate(test_dataset)
print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 529ms/step - accuracy: 0.6693 - loss: 1.9419
Test Accuracy : 0.6693280339241028
Test Loss : 1.941874384880066


In [ ]:
'''TEST ACCUARY ARE NOT BELIVABLE SO GONNA CHANGE THE DATA PREPROCESSING AND PIPLE LINE'''

In [15]:
data_augmentation_2 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
])

In [16]:
def preprocess_image2(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    return image, label

In [17]:
def preprocess_train2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    image = data_augmentation_2(image)
    return image, label

In [18]:
def preprocess_test2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    return image, label

In [19]:
train_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset2 = (
    train_dataset2
    .map(preprocess_train2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [20]:
test_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset2 = (
    test_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [21]:
val_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset2 = (
    val_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [22]:
ef_base_model3 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model3.trainable = False

efficientnet_model3 = models.Sequential([

    ef_base_model3,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model3.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [63]:
efficientnet_model3.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [64]:
history = efficientnet_model3.fit(train_dataset2,validation_data=val_dataset2,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 194s 839ms/step - accuracy: 0.0636 - loss: 1.9791 - val_accuracy: 0.0120 - val_loss: 1.9474
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 212s 935ms/step - accuracy: 0.0268 - loss: 1.9474 - val_accuracy: 0.0326 - val_loss: 1.9468
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 830ms/step - accuracy: 0.0675 - loss: 1.9463 - val_accuracy: 0.1098 - val_loss: 1.9470
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 178s 789ms/step - accuracy: 0.1077 - loss: 1.9473 - val_accuracy: 0.1098 - val_loss: 1.9497
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 178s 786ms/step - accuracy: 0.0545 - loss: 1.9462 - val_accuracy: 0.1098 - val_loss: 1.9492
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 192s 848ms/step - accuracy: 0.0990 - loss: 1.9463 - val_accuracy: 0.1098 - val_loss: 1.9477
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 180s 794ms/step - accuracy: 0.1097 - loss: 1.9462 - val_accuracy: 0.1098 - val_loss: 1.9460
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 180s 792ms/step - accuracy: 0.0916 - loss: 1.94

In [ ]:
'''FROM THE ABOVE MODEL WE CAN CONCLUDE THE MODEL PERFORM LESS WITH LOW DIMNESIONAL IMAGES IN NEXT GONNA PASS THE IMAGE RESOLUTION HIGH
EFFICENT NET MODEL BASICALLY WORK ON THE HIGH RESOLUTION TO CAPTURE HIGH FEATURE SO IN LAST MODEL NORMALIZING THE IMAGE IS AVOIDED

'''

In [31]:
ef_base_model4 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model4.trainable = False

efficientnet_model4 = models.Sequential([

    ef_base_model4,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model4.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [32]:
efficientnet_model4.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [75]:
history = efficientnet_model4.fit(train_dataset2,validation_data=val_dataset2,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 217s 949ms/step - accuracy: 0.4554 - loss: 1.4214 - val_accuracy: 0.4172 - val_loss: 1.3921
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 238s 1s/step - accuracy: 0.5678 - loss: 1.1336 - val_accuracy: 0.5795 - val_loss: 1.1057
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 221s 970ms/step - accuracy: 0.5985 - loss: 1.0218 - val_accuracy: 0.5329 - val_loss: 1.1441
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 198s 879ms/step - accuracy: 0.6041 - loss: 0.9743 - val_accuracy: 0.6055 - val_loss: 0.9990
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 203s 893ms/step - accuracy: 0.6409 - loss: 0.8854 - val_accuracy: 0.5802 - val_loss: 1.0594
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 189s 836ms/step - accuracy: 0.6413 - loss: 0.8846 - val_accuracy: 0.6447 - val_loss: 0.9180
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 178s 787ms/step - accuracy: 0.6549 - loss: 0.8240 - val_accuracy: 0.6401 - val_loss: 0.9562
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 817ms/step - accuracy: 0.6460 - loss: 0.8192 

In [ ]:
'''resnet model'''

In [77]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

resnet_model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

resnet_model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [78]:
resnet_model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [79]:
history = resnet_model.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 530s 2s/step - accuracy: 0.1304 - loss: 2.0004 - val_accuracy: 0.6693 - val_loss: 1.9328
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 514s 2s/step - accuracy: 0.5664 - loss: 1.9464 - val_accuracy: 0.6693 - val_loss: 1.9387
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 491s 2s/step - accuracy: 0.2919 - loss: 1.9463 - val_accuracy: 0.6693 - val_loss: 1.9423
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 481s 2s/step - accuracy: 0.2918 - loss: 1.9463 - val_accuracy: 0.1098 - val_loss: 1.9448
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 517s 2s/step - accuracy: 0.1109 - loss: 1.9464 - val_accuracy: 0.1111 - val_loss: 1.9458
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 517s 2s/step - accuracy: 0.0350 - loss: 1.9463 - val_accuracy: 0.0326 - val_loss: 1.9485
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 552s 2s/step - accuracy: 0.0327 - loss: 1.9463 - val_accuracy: 0.0326 - val_loss: 1.9481
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 525s 2s/step - accuracy: 0.0327 - loss: 1.9463 - val_accuracy: 0.032

In [25]:
image , label = next(iter(train_dataset))

print(image[0])


tf.Tensor(
[[[0.8080939  0.82091    0.79711413]
  [0.80837953 0.8212962  0.8026204 ]
  [0.80626655 0.82123303 0.7997037 ]
  ...
  [0.81725204 0.78602266 0.74610305]
  [0.8130437  0.7833483  0.74263096]
  [0.81110585 0.77942157 0.73636353]]

 [[0.8109243  0.8212929  0.8023424 ]
  [0.81119573 0.82422197 0.8037646 ]
  [0.8094835  0.81991625 0.80034995]
  ...
  [0.81763494 0.7884848  0.74748063]
  [0.8148469  0.785081   0.74358916]
  [0.8107054  0.78228104 0.7425555 ]]

 [[0.81164217 0.82240987 0.80637205]
  [0.81517243 0.8261093  0.8061037 ]
  [0.81604934 0.8201482  0.80243087]
  ...
  [0.81864655 0.78384995 0.7466042 ]
  [0.816923   0.7779974  0.74088025]
  [0.8143703  0.77987826 0.74435866]]

 ...

 [[0.797701   0.7739502  0.7580137 ]
  [0.7986597  0.7754518  0.75817585]
  [0.7980095  0.773661   0.75569725]
  ...
  [0.83084524 0.79125834 0.7792388 ]
  [0.8326061  0.7930677  0.77982223]
  [0.83403003 0.79436946 0.7807654 ]]

 [[0.79622936 0.776207   0.7571691 ]
  [0.7992532  0.7818353  0

In [27]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

resnet_model1 = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

resnet_model1.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [29]:
resnet_model1.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [30]:
history = resnet_model1.fit(train_dataset2,validation_data=val_dataset2,epochs=10,class_weight=class_weight_dict)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 513s 2s/step - accuracy: 0.3397 - loss: 1.7714 - val_accuracy: 0.4504 - val_loss: 1.3962
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 503s 2s/step - accuracy: 0.4353 - loss: 1.4746 - val_accuracy: 0.5675 - val_loss: 1.1242
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 487s 2s/step - accuracy: 0.4574 - loss: 1.3109 - val_accuracy: 0.5070 - val_loss: 1.2170
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 539s 2s/step - accuracy: 0.4872 - loss: 1.3023 - val_accuracy: 0.4285 - val_loss: 1.3603
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 558s 3s/step - accuracy: 0.5078 - loss: 1.2150 - val_accuracy: 0.5476 - val_loss: 1.1473
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 512s 2s/step - accuracy: 0.5106 - loss: 1.1879 - val_accuracy: 0.6454 - val_loss: 0.8923
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 531s 2s/step - accuracy: 0.5404 - loss: 1.1746 - val_accuracy: 0.5203 - val_loss: 1.1524
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 500s 2s/step - accuracy: 0.5323 - loss: 1.1404 - val_accuracy: 0.564

In [34]:
test_loss,test_accuracy = resnet_model1.evaluate(test_dataset)
print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.6693 - loss: 1.2162
Test Accuracy : 0.6693280339241028
Test Loss : 1.2162264585494995


In [ ]:
'''dense'''

In [40]:
densebase_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

densebase_model.trainable = False

dnmodel = tf.keras.Sequential([
    densebase_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7, activation="softmax")
])

dnmodel.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [41]:
dnmodel.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = dnmodel.fit(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=10,
    class_weight=class_weight_dict
)

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/week_7-ZVmopr-D/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 515s 2s/step - accuracy: 0.3115 - loss: 3.5742 - val_accuracy: 0.4365 - val_loss: 1.7415
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 490s 2s/step - accuracy: 0.2437 - loss: 1.8983 - val_accuracy: 0.2575 - val_loss: 1.7308
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 482s 2s/step - accuracy: 0.2861 - loss: 1.8785 - val_accuracy: 0.3952 - val_loss: 1.8255
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 495s 2s/step - accuracy: 0.2878 - loss: 1.8563 - val_accuracy: 0.4338 - val_loss: 1.6613
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 472s 2s/step - accuracy: 0.2946 - loss: 1.8264 - val_accuracy: 0.3965 - val_loss: 1.5755
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 539s 2s/step - accuracy: 0.2916 - loss: 1.8405 - val_accuracy: 0.2109 - val_loss: 1.7429
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 723s 3s/step - accuracy: 0.3113 - loss: 1.7699 - val_accuracy: 0.3999 - val_loss: 1.8094
Epoch 8/10
